# Fraud / Anomaly Detection — Provider Claims
Claim-line data → aggregate to TIN → z-score, IQR, Isolation Forest

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## 1. Load

In [ ]:
df = pd.read_csv('03_fraud_anomaly/claims_raw.csv', dtype=str)

print(df.shape)
df.dtypes

In [ ]:
# peek at the mess
df[['provider_tin', 'billed_amount', 'paid_amount', 'specialty', 'service_setting']].head(15)

In [ ]:
# value counts on key categoricals before cleaning
df['service_setting'].value_counts()

## 2. Clean

In [ ]:
# standardize TIN: strip all non-digits → 9-digit string
df = (
    df
    .assign(
        provider_tin = lambda x: x['provider_tin'].str.replace(r'[^0-9]', '', regex=True),
        billed_amount = lambda x: (
            x['billed_amount']
            .str.replace('$', '', fixed=True)
            .str.replace(',', '', fixed=True)
            .pipe(pd.to_numeric, errors='coerce')
        ),
        paid_amount = lambda x: (
            pd.to_numeric(x['paid_amount'], errors='coerce')
            .replace(-999, np.nan)       # sentinel → NaN
        ),
        procedure_code = lambda x: x['procedure_code'].str.strip().str.upper(),
        specialty      = lambda x: x['specialty'].str.strip().str.title(),
        service_setting = lambda x: x['service_setting'].str.strip().str.title(),
        member_id      = lambda x: pd.to_numeric(x['member_id'], errors='coerce'),
        claim_id       = lambda x: pd.to_numeric(x['claim_id'], errors='coerce'),
    )
)

print(df.shape)
df.isnull().sum()

In [ ]:
# confirm TINs are clean
df['provider_tin'].str.len().value_counts()

## 3. Aggregate to TIN level

In [ ]:
tin_df = (
    df
    .groupby('provider_tin', as_index=False)
    .agg(
        claim_count        = ('claim_id',     'count'),
        total_billed       = ('billed_amount', 'sum'),
        total_paid         = ('paid_amount',   'sum'),
        avg_billed_per_claim = ('billed_amount', 'mean'),
        unique_members     = ('member_id',     'nunique'),
        unique_procedures  = ('procedure_code', 'nunique'),
    )
    .assign(
        paid_to_billed_ratio = lambda x: x['total_paid'] / x['total_billed']
    )
)

print(tin_df.shape)
tin_df.describe()

## 4. Z-score flagging

In [ ]:
features = ['claim_count', 'total_billed', 'avg_billed_per_claim', 'unique_members']

z_scores = tin_df[features].apply(stats.zscore)
z_scores.columns = [f + '_z' for f in features]

tin_df = pd.concat([tin_df, z_scores], axis=1)

# flag if ANY feature has |z| > 3
tin_df['flag_zscore'] = (z_scores.abs() > 3).any(axis=1)

tin_df['flag_zscore'].value_counts()

## 5. IQR flagging

In [ ]:
iqr_flags = pd.DataFrame(index=tin_df.index)

for feat in features:
    q1  = tin_df[feat].quantile(0.25)
    q3  = tin_df[feat].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_flags[f'{feat}_iqr_flag'] = (tin_df[feat] < lower) | (tin_df[feat] > upper)

tin_df['flag_iqr'] = iqr_flags.any(axis=1)

tin_df['flag_iqr'].value_counts()

## 6. Isolation Forest

In [ ]:
# scale features first — IF doesn't require it but helps interpretation
X = tin_df[features].fillna(tin_df[features].median())
X_scaled = StandardScaler().fit_transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_scaled)

tin_df['flag_isoforest'] = iso.predict(X_scaled) == -1
tin_df['iso_score']       = iso.decision_function(X_scaled)  # lower = more anomalous

tin_df['flag_isoforest'].value_counts()

## 7. Compare methods

In [ ]:
# overlap table: how many TINs flagged by each combination
overlap = (
    tin_df[['flag_zscore', 'flag_iqr', 'flag_isoforest']]
    .astype(int)
    .assign(flagged_by = lambda x: x.sum(axis=1))
)

overlap['flagged_by'].value_counts().sort_index()

In [ ]:
# TINs flagged by all 3 methods — most confident outliers
all_three = tin_df[tin_df['flag_zscore'] & tin_df['flag_iqr'] & tin_df['flag_isoforest']]
print(f'Flagged by all 3 methods: {len(all_three)}')
all_three[['provider_tin', 'claim_count', 'total_billed', 'avg_billed_per_claim']].sort_values('total_billed', ascending=False)

In [ ]:
# scatter: claim_count vs avg_billed_per_claim, colored by flag status
flag_any = tin_df['flag_zscore'] | tin_df['flag_iqr'] | tin_df['flag_isoforest']

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(tin_df.loc[~flag_any, 'claim_count'],
           tin_df.loc[~flag_any, 'avg_billed_per_claim'],
           alpha=0.4, s=20, label='Normal', color='steelblue')
ax.scatter(tin_df.loc[flag_any, 'claim_count'],
           tin_df.loc[flag_any, 'avg_billed_per_claim'],
           alpha=0.8, s=50, label='Flagged (≥1 method)', color='crimson')
ax.set_xlabel('Claim Count')
ax.set_ylabel('Avg Billed per Claim ($)')
ax.set_title('TIN-level Anomaly Flags: Claim Volume vs. Avg Billed')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# method comparison bar chart
method_counts = pd.Series({
    'Z-score':          tin_df['flag_zscore'].sum(),
    'IQR':              tin_df['flag_iqr'].sum(),
    'Isolation Forest': tin_df['flag_isoforest'].sum(),
    'All 3':            (tin_df['flag_zscore'] & tin_df['flag_iqr'] & tin_df['flag_isoforest']).sum(),
})

fig, ax = plt.subplots(figsize=(7, 4))
method_counts.plot(kind='bar', ax=ax, color=['steelblue','steelblue','steelblue','crimson'], rot=0)
ax.set_ylabel('TINs Flagged')
ax.set_title('Flags by Detection Method')
plt.tight_layout()
plt.show()